# Phase 3: Hardware-Accelerated Model Fine-Tuning

## Executive Overview
This module instantiates a 106-million parameter foundation model (Geneformer) and fine-tunes it to predict continuous transcriptomic perturbation effects. 

To maximize raw VRAM efficiency and GPU throughput, this architecture bypasses standard Hugging Face wrappers and directly integrates with **NVIDIA's TransformerEngine (TE)**. Because TE strips away secondary masking arguments to optimize C++ CUDA kernels, we utilize a custom **Layer Unrolling** technique to manually execute the forward pass, sidestepping framework interface mismatches.

### Key Objectives
1. **TransformerEngine Integration:** Route native sequence embeddings through NVIDIA's accelerated FP8-capable BERT backbone.
2. **Layer Unrolling Bypass:** Manually construct hardware-level attention masks and loop through encoder matrices to resolve Hugging Face `return_dict` limitations.
3. **Continuous Regression Head:** Extract the `[CLS]` cell-state token and project it via an `nn.Linear` head to predict the perturbation dose-response proxy (`num_features`).
4. **Gradient Optimization:** Execute an AdamW-driven training loop across the deserialized single-cell rank-value arrays.

In [ ]:
# =====================================================================
# GLOBAL PIPELINE BOUNDARY CONFIGURATION
# =====================================================================
import os
import sys

# IO Pathing
INPUT_DIRECTORY  = "data"
INPUT_FILENAME   = "geneformer_tuning_input.pkl"
OUTPUT_DIRECTORY = "data"
OUTPUT_FILENAME  = "gout_geneformer_finetuned.pt"

# Framework Pathing (BioNeMo internal recipes)
BIONEMO_RECIPE_PATH = "/bionemo-framework/bionemo-recipes/models/geneformer/src/geneformer"

# Model Hyperparameters
BATCH_SIZE    = 4        # Micro-batching to respect memory envelopes
LEARNING_RATE = 5e-5
WEIGHT_DECAY  = 0.01
NUM_EPOCHS    = 3

input_handoff_path = os.path.join(INPUT_DIRECTORY, INPUT_FILENAME)
output_checkpoint_path = os.path.join(OUTPUT_DIRECTORY, OUTPUT_FILENAME)

print("Hyperparameters and architectural paths locked.")

## Framework Pathing & Architecture Instantiation
We link the Python interpreter directly to the native TransformerEngine recipes and define the unrolled neural network architecture.

In [ ]:
import torch
import torch.nn as nn

# Force CUDA to execute commands synchronously for precise error tracing
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# Link to NVIDIA native recipes
if BIONEMO_RECIPE_PATH not in sys.path:
    sys.path.append(BIONEMO_RECIPE_PATH)

from modeling_bert_te import BertModel, TEBertConfig

# 1. Initialize strict hardware configuration map
model_config = TEBertConfig(
    hidden_size=768,
    num_attention_heads=12,
    num_hidden_layers=6,
    vocab_size=40000,
    max_position_embeddings=2048
)
model_config.position_embedding_type = "absolute"
model_config.is_decoder = False

# 2. Define the Unrolled Architecture Wrapper
class GeneformerForRegression(nn.Module):
    def __init__(self, config):
        super().__init__()
        # Load the base NVIDIA model (Bypassing the default forward wrapper)
        self.bert = BertModel(config)
        self.regression_head = nn.Linear(config.hidden_size, 1)

    def forward(self, input_ids):
        # Step 1: Extract initial embeddings natively
        hidden_states = self.bert.embeddings(input_ids=input_ids)

        # Step 2: Construct hardware-level Attention Mask (Batch, 1, 1, Seq)
        batch_size, seq_length = input_ids.shape
        attention_mask = torch.zeros(
            (batch_size, 1, 1, seq_length),
            dtype=hidden_states.dtype,
            device=hidden_states.device
        )

        # Step 3: Execution Bypass (Unrolling)
        for layer_module in self.bert.encoder.layer:
            layer_outputs = layer_module(hidden_states, attention_mask)
            # Handle Tuple outputs natively generated by TransformerEngine
            hidden_states = layer_outputs[0] if isinstance(layer_outputs, tuple) else layer_outputs

        # Step 4: Extract cell representation (Position 0 = [CLS] Token)
        cell_embedding = hidden_states[:, 0, :]

        # Step 5: Final continuous dosage prediction
        return self.regression_head(cell_embedding)

regression_model = GeneformerForRegression(model_config)
print("✅ Success: 'regression_model' instantiated with Unrolled execution bypass.")

## Data Loading & Tensor Packaging
We deserialize the rank-ordered arrays from Phase 2 and construct a PyTorch `DataLoader` to feed the GPU memory banks efficiently via micro-batching.

In [ ]:
import pickle
from torch.utils.data import DataLoader, Dataset

class GoutDosageDataset(Dataset):
    def __init__(self, data_path):
        with open(data_path, 'rb') as f:
            data = pickle.load(f)
        self.tokens = data['tokens']
        self.dosage = data['dosage']

    def __len__(self):
        return len(self.tokens)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.tokens[idx], dtype=torch.long),
            "target": torch.tensor([self.dosage[idx]], dtype=torch.float)
        }

train_dataset = GoutDosageDataset(input_handoff_path)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True 
)

print(f"Success: Conveyor belt initialized with {len(train_dataset)} single-cell arrays.")

## The Gradient Optimization Loop
We execute the forward passes, compute the Mean Squared Error (MSE), and backpropagate the weight adjustments into the self-attention heads utilizing the **AdamW** optimizer.

In [ ]:
from torch.optim import AdamW

# 1. Hardware Routing
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training environment routed to: {device}")
regression_model.to(device)

# 2. Optimization Settings
optimizer = AdamW(regression_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.MSELoss() 

# 3. Execution Loop
print(f"\nInitiating Training Loop for {NUM_EPOCHS} Epochs...")

for epoch in range(NUM_EPOCHS):
    regression_model.train() 
    total_epoch_loss = 0.0

    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        targets = batch["target"].to(device)

        # Clear previous gradients
        optimizer.zero_grad() 

        # Forward Pass
        predictions = regression_model(input_ids)

        # Compute Error
        loss = criterion(predictions, targets)

        # Backward Pass (Calculate Gradients)
        loss.backward()

        # Optimization Step (Update Weights)
        optimizer.step()

        total_epoch_loss += loss.item()

    avg_loss = total_epoch_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | Average Training MSE Loss: {avg_loss:.4f}")

print("\nModel Fine-Tuning Sequence Completed Successfully.")

## Model State Serialization
The fine-tuned mathematical weights are extracted from active VRAM and persisted to disk as a `.pt` file, allowing us to deploy the model in an isolated inference environment without retraining.

In [ ]:
os.makedirs(OUTPUT_DIRECTORY, exist_ok=True)

# Extract and save the model's internal learned weights (State Dictionary)
torch.save(regression_model.state_dict(), output_checkpoint_path)

print(f"Milestone Saved: Fine-tuned model checkpoint securely written to {output_checkpoint_path}")